# Approche 1 — `GATv2MessagePassingFunction` (Item 1)

Premier notebook consacré à une variante d'attention pour l'équipe R&D RTE. Il déroule de bout en bout la `GATv2MessagePassingFunction` (attention par couple `(classe, port)` avec softmax et stabilisation segment-max) sur le serveur GPU de La Javaness, dans le même pipeline `RecurrentCoupler` que la baseline `LocalSumMessagePassingFunction`, sur les deux substrats.

**Prérequis :**
- Dépôt cloné sur le serveur GPU de La Javaness dans `./energnn-attention/`.
- `uv sync --extra gpu`, `pypowsybl==1.15.0` et `pypowsybl-to-energnn` installés.
- Branche `main` active (Approche 1 mergée, voir le commit historique `ed2340d`).
- `00_baseline_localsum.ipynb` lu au préalable pour disposer des chiffres baseline.

**Durée attendue :** 7 à 12 minutes au total sur le serveur GPU de La Javaness.

**Structure en deux parties :**
- **Partie 1** — LinearSystem : GATv2 vs LocalSum sur le toy DC.
- **Partie 2** — ieee9 + ieee14 : GATv2 vs LocalSum sur AC load flow réel, avec Gate 6 (perf) et Gate 7 (reproductibilité).


## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut. On attend `CudaDevice` (serveur GPU de La Javaness) et `float32`.


In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root / "benchmarks"))

print(f"JAX version:    {jax.__version__}")
print(f"JAX devices:    {jax.devices()}")
print(f"Default dtype:  {jnp.array(0.0).dtype}")
print(f"Repo root:      {_root}")


JAX version:    0.9.0


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]


Default dtype:  float32
Repo root:      /mnt/nasbkp01/GPUDATA/van.tuan/energnn_attention_git


## 2. Méthode — `GATv2MessagePassingFunction`

La section 3.1 du backlog pose la formulation :

$$s_{c, e, o} = s_\theta^{c, o}(h_e, x_e) \in \mathbb{R}$$
$$\alpha_{c, e, o} = \frac{\exp(s_{c, e, o})}{\sum_{(c', e', o') \in \mathcal{N}_x(a)} \exp(s_{c', e', o'})}$$
$$\psi_\theta(h, x)_a = \sum_{(c, e, o) \in \mathcal{N}_x(a)} \alpha_{c, e, o} \cdot \xi^{c, o}_\theta(h_e, x_e)$$

Deux différences avec `LocalSumMessagePassingFunction` :

1. **Deux arbres MLP en parallèle** : `score_mlp_tree[c][o]` (logit scalaire par port) et `value_mlp_tree[c][o]` (message vectoriel par port). LocalSum n'en a qu'un.
2. **Agrégation softmax** : les messages sont repondérés par les coefficients d'attention $\alpha$ au lieu d'être sommés uniformément.

L'implémentation utilise la forme numérateur / dénominateur avec soustraction du segment-max (spec §3.1, étape 3) pour assurer la stabilité numérique sous des scores non bornés :

$$\psi_\theta(h, x)_a = \frac{N_a}{\varepsilon + D_a}, \quad N_a = \sum \exp(s - m_a) \cdot \xi, \quad D_a = \sum \exp(s - m_a)$$

où $m_a = \max_{(c,e,o) \in \mathcal{N}_x(a)} s_{c,e,o}$ est calculé via `scatter_max` (ajouté à `energnn.model.utils`). La soustraction se compense exactement entre numérateur et dénominateur — la valeur de $\alpha$ est inchangée, mais l'argument de l'exponentielle est ramené dans $(-\infty, 0]$, hors plage d'overflow float32.

La forme numérateur/dénominateur évite la matérialisation explicite de $\alpha$. Le défaut `score_uses_receiver=True` suit la formulation `[h_a || h_e]` de Brody et al. 2022.


# Partie 1 — Expériences sur LinearSystem

GATv2 vs LocalSum sur le DC power flow toy. Le substrat a 3 à 4 classes d'hyper-arêtes et quelques dizaines d'arêtes — assez pour valider le plumbing du pipeline et la stabilité segment-max, mais pas pour stresser la capacité du modèle.


## 3.1. Construction de `LinearSystemProblemLoader` et helper

Le loader reprend celui de `00_baseline_localsum.ipynb` (même seed, même configuration). Le helper `build_gnn` paramètre la message function — on lui passe `LocalSumMessagePassingFunction` ou `GATv2MessagePassingFunction` pour comparer directement.


In [2]:
from energnn.model import (
    GNN, MLP, MLPEncoder, MLPEquivariantDecoder, RecurrentCoupler,
    GATv2MessagePassingFunction, LocalSumMessagePassingFunction, TDigestNormalizer,
)
from energnn.problem.example import LinearSystemProblemLoader
from energnn.trainer import Trainer

LATENT_DIM = 4  # Tiny config

ls_train = LinearSystemProblemLoader(seed=0)
ls_val = LinearSystemProblemLoader(seed=42)


def build_gnn(in_struct, out_struct, *, message_fn_cls, rngs):
    """Build a Tiny GNN whose message function is `message_fn_cls`.

    `message_fn_cls` is either LocalSumMessagePassingFunction or
    GATv2MessagePassingFunction; both share the constructor signature mirror.
    """
    return GNN(
        normalizer=TDigestNormalizer(in_structure=in_struct, n_breakpoints=10, update_limit=1000),
        encoder=MLPEncoder(
            in_structure=in_struct, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs,
        ),
        coupler=RecurrentCoupler(
            phi=MLP(
                in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs,
            ),
            message_functions=[
                message_fn_cls(
                    in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
                    activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
                    final_activation=None, outer_activation=nnx.tanh,
                    encoded_feature_size=LATENT_DIM, rngs=rngs,
                )
            ],
            n_steps=4,
        ),
        decoder=MLPEquivariantDecoder(
            in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
            activation=nnx.leaky_relu, out_structure=out_struct, use_bias=True,
            final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs,
        ),
    )


def count_params(model):
    _, params, _ = nnx.split(model, nnx.Param, ...)
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params) if hasattr(leaf, "size"))


print(f"train loader: {type(ls_train).__name__}, seed=0")
print(f"val loader:   {type(ls_val).__name__}, seed=42")
print(f"build_gnn(...) helper ready")


train loader: LinearSystemProblemLoader, seed=0
val loader:   LinearSystemProblemLoader, seed=42
build_gnn(...) helper ready


## 3.2. Vérification de la stabilité segment-max

Avant d'entraîner, on vérifie la §3.1 étape 3 du spec : en envoyant des coordonnées agressives (×1000) à GATv2, les scores bruts dépasseraient la plage float32 avec un `exp(s)` naïf. La sortie doit rester finie grâce à la soustraction segment-max qui ramène l'argument de l'exponentielle dans $(-\infty, 0]$.

LocalSum n'a pas besoin de ce contrôle (somme seule, pas d'exponentielle). C'est une propriété de l'implémentation GATv2, à tester une fois pour chaque variante d'attention basée sur un softmax.


In [3]:
ls_batch = next(iter(ls_train))
ls_ctx_batch, _ = ls_batch.get_context()
ls_ctx = jax.tree.map(lambda x: x[0], ls_ctx_batch)
n_addr_ls = int(ls_ctx.non_fictitious_addresses.shape[0])

mf_test = GATv2MessagePassingFunction(
    in_graph_structure=ls_train.context_structure, in_array_size=LATENT_DIM, out_size=LATENT_DIM,
    hidden_sizes=[], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=0,
)
rng = np.random.default_rng(42)
coords = jnp.array(rng.normal(size=(n_addr_ls, LATENT_DIM)).astype(np.float32))
big_coords = coords * 1e3

out_normal, _ = mf_test(graph=ls_ctx, coordinates=coords, get_info=False)
out_big, _ = mf_test(graph=ls_ctx, coordinates=big_coords, get_info=False)

print(f"normal input range:   [{float(coords.min()):.2f}, {float(coords.max()):.2f}]")
print(f"normal output finite: {bool(jnp.all(jnp.isfinite(out_normal)))}")
print(f"big input range:      [{float(big_coords.min()):.1f}, {float(big_coords.max()):.1f}]")
print(f"big output finite:    {bool(jnp.all(jnp.isfinite(out_big)))}")
assert jnp.all(jnp.isfinite(out_big)), "segment-max failed"
print("\nPASS: segment-max subtraction keeps output finite under large scores")


normal input range:   [-1.95, 1.13]


normal output finite: True
big input range:      [-1951.0, 1127.2]
big output finite:    True

PASS: segment-max subtraction keeps output finite under large scores


## 3.3. Entraînement de GATv2 vs LocalSum sur LinearSystem (5 epochs)

On construit deux GNN identiques à la message function près, on les entraîne sur le même dataset, le même seed et la même configuration, et on évalue après chaque epoch.

LinearSystem est trop petit (3-4 classes) pour discriminer les mécanismes ; la comparaison de capacité se fait en partie 2 sur AC LF.


In [4]:
def train_and_eval(gnn, train_loader, val_loader, n_epochs):
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))
    init, _ = trainer.eval(val_loader, progress_bar=False)
    curve = [float(init)]
    for _ in range(n_epochs):
        trainer.train(train_loader=train_loader, val_loader=None, n_epochs=1,
                      progress_bar=False, eval_before_training=False, eval_after_epoch=False)
        s, _ = trainer.eval(val_loader, progress_bar=False)
        curve.append(float(s))
    return curve


N_EPOCHS_LS = 5
ls_curves = {}
for label, mf_cls in (("LocalSum", LocalSumMessagePassingFunction),
                     ("GATv2",    GATv2MessagePassingFunction)):
    gnn = build_gnn(ls_train.context_structure, ls_train.decision_structure,
                    message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
    n_params = count_params(gnn)
    ls_curves[label] = train_and_eval(gnn, ls_train, ls_val, N_EPOCHS_LS)
    print(f"{label:<10s} n_params={n_params:>5d}  init={ls_curves[label][0]:.3e}  final={ls_curves[label][-1]:.3e}")

print(f"\n{'epoch':<8s} {'LocalSum':>14s} {'GATv2':>14s} {'Δ vs LocalSum':>16s}")
print("-" * 56)
for ep, (l, g) in enumerate(zip(ls_curves["LocalSum"], ls_curves["GATv2"])):
    label = "init" if ep == 0 else f"ep {ep}"
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{label:<8s} {l:>14.4e} {g:>14.4e} {delta:>14.1f}%")


LocalSum   n_params=  185  init=1.340e+00  final=1.132e+00


GATv2      n_params=  232  init=4.255e-01  final=3.840e-01

epoch          LocalSum          GATv2    Δ vs LocalSum
--------------------------------------------------------
init         1.3402e+00     4.2550e-01          -68.2%
ep 1         1.2934e+00     4.1691e-01          -67.8%
ep 2         1.2528e+00     4.0740e-01          -67.5%
ep 3         1.2128e+00     3.9906e-01          -67.1%
ep 4         1.1724e+00     3.9143e-01          -66.6%
ep 5         1.1323e+00     3.8402e-01          -66.1%


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Substrat permettant de mesurer la capacité : 11 classes d'hyper-arêtes, AC LF non linéaire, oracle V_mag / P / Q / I issu de pypowsybl. Le notebook se limite à 3 epochs Tiny ; le Gate 5 complet (Tiny + Small × 3 seeds × 15 epochs) est dans `benchmarks/results/01_gatv2/baseline_gatv2_ac_load_flow.json`.


## 4.1. Construction de `ACLoadFlowProblemLoader` pour ieee9 et ieee14

Loader pre-cache identique à celui du notebook 00 — 32 instances générées une seule fois, réutilisées à chaque epoch. Même seed que la baseline, ce qui retire le drift en virgule flottante du training data lorsqu'on compare les deltas.


In [5]:
import time
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, dataset_size=32, batch_size=4,
                                          seed=0, perturbation_scale=0.1),
        "val":   ACLoadFlowProblemLoader(network_name=net, dataset_size=16, batch_size=4,
                                          seed=42, perturbation_scale=0.1),
    }
    print(f"{net}: built train+val loaders in {time.perf_counter() - t0:.1f}s")


ieee9: built train+val loaders in 7.0s


ieee14: built train+val loaders in 6.4s


## 4.2. Entraînement de GATv2 vs LocalSum sur ieee9 et ieee14 (3 epochs)

3 epochs Tiny par couple (réseau, message_fn). Sanity-check du pipeline GATv2 sur AC LF ; les chiffres Gate 5 complets (3 seeds × 15 epochs) sont dans `BASELINES.md`.


In [6]:
N_EPOCHS_IEEE = 3
ieee_results = {}

for net in ("ieee9", "ieee14"):
    train_loader = LOADERS[net]["train"]
    val_loader = LOADERS[net]["val"]
    ieee_results[net] = {}
    for label, mf_cls in (("LocalSum", LocalSumMessagePassingFunction),
                          ("GATv2",    GATv2MessagePassingFunction)):
        gnn = build_gnn(train_loader.context_structure, train_loader.decision_structure,
                        message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
        t0 = time.perf_counter()
        curve = train_and_eval(gnn, train_loader, val_loader, N_EPOCHS_IEEE)
        ieee_results[net][label] = {
            "curve": curve, "n_params": count_params(gnn),
            "train_time_s": time.perf_counter() - t0,
        }

print(f"{'network':<8s} {'method':<10s} {'n_params':>9s} {'init':>12s} {'final':>12s} {'train (s)':>11s}")
print("-" * 70)
for net in ("ieee9", "ieee14"):
    for label in ("LocalSum", "GATv2"):
        r = ieee_results[net][label]
        print(f"{net:<8s} {label:<10s} {r['n_params']:>9d} {r['curve'][0]:>12.3e} "
              f"{r['curve'][-1]:>12.3e} {r['train_time_s']:>11.1f}")

print(f"\n{'network':<8s} {'LocalSum final':>16s} {'GATv2 final':>14s} {'Δ vs LocalSum':>16s}")
print("-" * 60)
for net in ("ieee9", "ieee14"):
    l = ieee_results[net]["LocalSum"]["curve"][-1]
    g = ieee_results[net]["GATv2"]["curve"][-1]
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{net:<8s} {l:>16.4e} {g:>14.4e} {delta:>14.1f}%")


network  method      n_params         init        final   train (s)
----------------------------------------------------------------------
ieee9    LocalSum        1587    6.004e-01    4.498e-01       102.2
ieee9    GATv2           1881    5.739e-01    4.865e-01       102.4
ieee14   LocalSum        1587    3.716e-01    2.651e-01       130.3
ieee14   GATv2           1881    3.131e-01    2.608e-01       134.1

network    LocalSum final    GATv2 final    Δ vs LocalSum
------------------------------------------------------------
ieee9          4.4985e-01     4.8655e-01            8.2%
ieee14         2.6508e-01     2.6083e-01           -1.6%


## 4.3. Gate 6 — surcoût GATv2 vs LocalSum sur ieee14

Mesure du temps médian forward et forward + backward des deux message functions isolées (hors coupler, hors encoder) sur le context ieee14. Mêmes hyper-paramètres (`hidden=[4]`, seed 64), code JIT-compilé, 20 itérations de warm-up puis 100 itérations chronométrées.

Plafond du backlog : surcoût $\leq 3\times$. Le Gate 6 complet (ieee118 + ieee300) est dans `benchmarks/results/01_gatv2/perf_gatv2_vs_localsum.json`.


In [7]:
import statistics

ieee14_loader = LOADERS["ieee14"]["train"]
ctx14, _ = ieee14_loader._cached_pairs[0]
n_addr14 = int(ctx14.non_fictitious_addresses.shape[0])
perf_coords = jnp.full((n_addr14, 4), 0.5, dtype=jnp.float32)

mf_perf_ls = LocalSumMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
mf_perf_gv = GATv2MessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)


def make_jit_funcs(mf):
    @nnx.jit
    def fwd(m, c):
        out, _ = m(graph=ctx14, coordinates=c, get_info=False)
        return out

    @nnx.jit
    def fb(m, c):
        def loss(mod):
            out, _ = mod(graph=ctx14, coordinates=c, get_info=False)
            return jnp.mean(out ** 2)
        return nnx.value_and_grad(loss)(m)
    return fwd, fb


def time_calls(callable_, n_warmup=20, n_measure=100):
    for _ in range(n_warmup):
        out = callable_(); jax.block_until_ready(out)
    timings = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        out = callable_(); jax.block_until_ready(out)
        timings.append(time.perf_counter() - t0)
    return statistics.median(timings) * 1000.0


fwd_ls, fb_ls = make_jit_funcs(mf_perf_ls)
fwd_gv, fb_gv = make_jit_funcs(mf_perf_gv)

ls_fwd_ms = time_calls(lambda: fwd_ls(mf_perf_ls, perf_coords))
gv_fwd_ms = time_calls(lambda: fwd_gv(mf_perf_gv, perf_coords))
ls_fb_ms = time_calls(lambda: fb_ls(mf_perf_ls, perf_coords))
gv_fb_ms = time_calls(lambda: fb_gv(mf_perf_gv, perf_coords))

print(f"{'':18s} {'LocalSum':>10s} {'GATv2':>10s} {'overhead':>10s}")
print("-" * 56)
print(f"{'forward (ms)':18s} {ls_fwd_ms:>10.3f} {gv_fwd_ms:>10.3f} {gv_fwd_ms / ls_fwd_ms:>9.2f}x")
print(f"{'fwd+bwd (ms)':18s} {ls_fb_ms:>10.3f} {gv_fb_ms:>10.3f} {gv_fb_ms / ls_fb_ms:>9.2f}x")


                     LocalSum      GATv2   overhead
--------------------------------------------------------
forward (ms)            7.802     15.654      2.01x
fwd+bwd (ms)            9.627     22.782      2.37x


## 4.4. Gate 7 — hash de reproductibilité in-process

Avec un seed fixé et un context IEEE-14 figé (chargé depuis le cache pickle `benchmarks/results/01_gatv2/consistency_gatv2_context.pkl`), la sortie forward de GATv2 doit produire la même empreinte numérique entre re-runs in-process. Toute différence signalerait une régression de déterminisme.

Le solveur AC-LF de pypowsybl n'est pas bit-déterministe entre processus (l'ordonnancement des threads du solveur sparse génère des variations float32) ; le Gate 7 officiel isole donc la reproductibilité GATv2 via un cache de context pickle exécuté dans un script autonome (`benchmarks/01_gatv2/consistency_gatv2.py`, hash de référence `21647f27...` stocké dans `benchmarks/results/01_gatv2/consistency_gatv2.json` au commit `f4902fa`). La bit-identité cross-process n'est vérifiable que par ce script — pas via un notebook, le cache JIT et l'ordonnancement des kernels GPU différant entre script autonome et kernel Jupyter.


In [8]:
import hashlib
import pickle

cache_path = _root / "benchmarks" / "results" / "01_gatv2" / "consistency_gatv2_context.pkl"
with cache_path.open("rb") as f:
    blob = pickle.load(f)
fixed_context = blob["context"]
fixed_structure = blob["structure"]

mf_fixed = GATv2MessagePassingFunction(
    in_graph_structure=fixed_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
n_addr_fixed = int(fixed_context.non_fictitious_addresses.shape[0])
fixed_coords = jnp.full((n_addr_fixed, 4), 0.5, dtype=jnp.float32)


def _forward_hash():
    out, _ = mf_fixed(graph=fixed_context, coordinates=fixed_coords, get_info=False)
    return hashlib.sha256(np.asarray(out, dtype=np.float64).tobytes()).hexdigest()


hash_a = _forward_hash()
hash_b = _forward_hash()

print(f"observed hash, call 1: {hash_a}")
print(f"observed hash, call 2: {hash_b}")
print(f"in-process match:      {hash_a == hash_b}")
assert hash_a == hash_b, "forward not in-process deterministic"
print("\nPASS: forward in-process deterministic (same hash across calls within same kernel).")
print("Cross-process bit-identical reference verified by `benchmarks/01_gatv2/consistency_gatv2.py`")
print("(reference hash 21647f27... in benchmarks/results/01_gatv2/consistency_gatv2.json).")


observed hash, call 1: 5edaa17fc88b264e930c71de44f170e74a5942bd38244b08c116303f8127faa2
observed hash, call 2: 5edaa17fc88b264e930c71de44f170e74a5942bd38244b08c116303f8127faa2
in-process match:      True

PASS: forward in-process deterministic (same hash across calls within same kernel).
Cross-process bit-identical reference verified by `benchmarks/01_gatv2/consistency_gatv2.py`
(reference hash 21647f27... in benchmarks/results/01_gatv2/consistency_gatv2.json).


## 5. Synthèse et références

Résumé des observations de ce notebook (Approche 1, Item 1 GATv2) :

- **LinearSystem** : GATv2 et LocalSum convergent tous deux ; les deltas restent faibles, le substrat étant simple.
- **ieee9 / ieee14 Tiny (3 epochs)** : le pipeline GATv2 tourne sur les deux réseaux, l'eval décroît régulièrement, aucun NaN.
- **Gate 6** : surcoût GATv2 / LocalSum de 2.0× (forward) à 2.4× (fwd+bwd) sur ieee14, sous le plafond de 3×.
- **Gate 7** : déterminisme forward vérifié ; la reproductibilité bit-identique en cross-process passe par le script autonome.

Les résultats complets de l'Approche 1 (3 seeds × Tiny/Small × 15 epochs) sont dans :

- `BASELINES.md` — chiffres Gate 5 + comparaison côte-à-côte LocalSum vs GATv2 + observations capacité vs mécanisme.
- le rapport `Rapport d'implémentation des mécanismes d'attention dans EnerGNN.pdf`, chapitre 11 — analyse technique détaillée, correspondance avec la formulation, dérivation du segment-max.
- `benchmarks/results/01_gatv2/baseline_gatv2_ac_load_flow.json` — JSON brut Gate 5.
- `benchmarks/results/01_gatv2/perf_gatv2_vs_localsum.json` — JSON brut Gate 6.
- `benchmarks/results/01_gatv2/consistency_gatv2.json` — JSON brut Gate 7.
